In [5]:
program pendulum_simulation
    implicit none
    integer, parameter :: n_steps = 200
    real(8), parameter :: dt = 0.04_8
    real(8), parameter :: g = 9.81_8
    real(8), parameter :: length = 1.0_8
    real(8), parameter :: pi = 3.14159265358979_8

    real(8) :: theta, omega
    real(8) :: k1_theta, k1_omega
    real(8) :: k2_theta, k2_omega
    real(8) :: k3_theta, k3_omega
    real(8) :: k4_theta, k4_omega
    real(8) :: x_pos, y_pos
    integer :: step, unit_num
    character(len=30) :: filename

    theta = pi / 3.0_8
    omega = 0.0_8

    call execute_command_line("mkdir -p pendulum_frames")

    do step = 1, n_steps
        x_pos = length * sin(theta)
        y_pos = -length * cos(theta)

        write(filename, '(A, I0, A)') "pendulum_frames/f", step, ".dat"
        open(newunit=unit_num, file=trim(filename), status='replace')
        write(unit_num, '(F10.5, 1X, F10.5)') 0.0_8, 0.0_8
        write(unit_num, '(F10.5, 1X, F10.5)') x_pos, y_pos
        close(unit_num)

        k1_theta = omega
        k1_omega = -(g / length) * sin(theta)

        k2_theta = omega + 0.5_8 * dt * k1_omega
        k2_omega = -(g / length) * sin(theta + 0.5_8 * dt * k1_theta)

        k3_theta = omega + 0.5_8 * dt * k2_omega
        k3_omega = -(g / length) * sin(theta + 0.5_8 * dt * k2_theta)

        k4_theta = omega + dt * k3_omega
        k4_omega = -(g / length) * sin(theta + dt * k3_theta)

        theta = theta + (dt / 6.0_8) * (k1_theta + 2.0_8*k2_theta + 2.0_8*k3_theta + k4_theta)
        omega = omega + (dt / 6.0_8) * (k1_omega + 2.0_8*k2_omega + 2.0_8*k3_omega + k4_omega)
    end do

    print *, "Pendulum frames written to pendulum_frames/"
end program pendulum_simulation

 Pendulum frames written to pendulum_frames/


In [7]:
program generate_animation
    implicit none
    integer :: unit_num, i
    integer, parameter :: n_steps = 200
    character(len=100) :: line

    open(newunit=unit_num, file='pendulum_animate.gp', status='replace')
    write(unit_num, '(A)') "set terminal gif animate delay 4 size 500,500"
    write(unit_num, '(A)') "set output 'pendulum.gif'"
    write(unit_num, '(A)') "set xrange [-1.2:1.2]"
    write(unit_num, '(A)') "set yrange [-1.2:0.2]"
    write(unit_num, '(A)') "unset key"
    write(unit_num, '(A)') "set size square"

    do i = 1, n_steps
        write(line, '(A, I0, A)') "plot 'pendulum_frames/f", i, ".dat' with linespoints lw 2 pt 7 ps 3"
        write(unit_num, '(A)') trim(line)
    end do

    close(unit_num)

    call execute_command_line("gnuplot pendulum_animate.gp")

    print *, "Animated GIF saved to pendulum.gif"
end program generate_animation

200 frames in animation sequence


 Animated GIF saved to pendulum.gif


![Pendulum animation](pendulum.gif)